**KOREKSI SAMPLING EFFORT — LUAS LAHAN TERBANGUN DYNAMIC WORLD Provinsi D.I. Yogyakarta, 2016-2025**

**LATAR BELAKANG**

Agregasi tahunan Dynamic World menghasilkan estimasi luas terbangun yang
dipengaruhi banyaknya observasi Sentinel-2 yang tersedia pada tahun tersebut. Piksel permukiman di Jawa merupakan campuran atap, pohon, dan halaman; semakin banyak observasi, semakin besar peluang piksel tersebut
terklasifikasi sebagai terbangun. Akibatnya luas terukur ikut naik-turun
mengikuti ketersediaan citra, bukan perubahan lahan yang sesungguhnya.

**METODE**

Penyesuaian kovariat (covariate adjustment / ANCOVA-style correction),
setara dengan koreksi sampling effort yang lazim dipakai ketika intensitas pengamatan tidak seragam antar periode.

    Luas = a + b * Obs                                   ... model
    Luas_terkoreksi = Luas - b * (Obs - Obs_acuan)       ... penyesuaian

**SYARAT KEABSAHAN**

Jumlah observasi harus tidak berkorelasi dengan tahun. Bila berkorelasi koreksi akan ikut menghilangkan tren temporal yang sesungguhnya. Skrip ini mengujinya secara eksplisit.

In [9]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
import numpy as np
import pandas as pd
from scipy import stats

KONFIGURASI

In [11]:
PATH_DW = "/content/drive/MyDrive/Skripsi_Citra/Dynamic World - Built Up Area/Perbandingan Jendela Citra/DW_Terbangun_TanpaMask_DIY_2016_2025.csv"
KOLOM_LUAS = "luas_lunak_ha"        # metode agregasi yang dipakai
KOLOM_OBS  = "obs_per_piksel"
LUAS_DIY   = 318_580                # hektar, BPS

1. MEMBACA DATA

In [12]:
df = pd.read_csv(PATH_DW)
df["tahun"] = df["tahun"].astype(int)
df = df.sort_values("tahun").reset_index(drop=True)

y   = df[KOLOM_LUAS].values         # luas terbangun (ha)
obs = df[KOLOM_OBS].values          # observasi per piksel
th  = df["tahun"].values

print("=" * 74)
print("DATA MASUKAN")
print("=" * 74)
print(df[["tahun", "n_citra", KOLOM_OBS, "cakupan_ha", KOLOM_LUAS]]
      .round(1).to_string(index=False))

DATA MASUKAN
 tahun  n_citra  obs_per_piksel  cakupan_ha  luas_lunak_ha
  2016       53             4.3    270331.9        53500.4
  2017       97             7.0    315857.8        62526.7
  2018      194            16.2    315871.1        67507.8
  2019      244            22.4    315871.1        73595.3
  2020      155            13.5    315871.1        68308.9
  2021      142             8.5    315519.5        67656.4
  2022      110             7.1    315797.8        65023.8
  2023      178            16.7    315871.1        79881.1
  2024      188            15.2    315871.1        77437.3
  2025      152             9.3    315871.1        65018.5


2. UJI SYARAT: apakah observasi berkorelasi dengan tahun?

In [13]:
print()
print("=" * 74)
print("UJI SYARAT KEABSAHAN KOREKSI")
print("=" * 74)

r_ot, p_ot = stats.pearsonr(th, obs)
print(f"Korelasi observasi vs tahun : r = {r_ot:+.3f}, p = {p_ot:.3f}")
if p_ot > 0.05:
    print("  -> tidak signifikan. Observasi tidak mengikuti tren waktu,")
    print("     sehingga koreksi tidak akan menghilangkan tren temporal asli.")
    print("     SYARAT TERPENUHI.")
else:
    print("  -> SIGNIFIKAN. Observasi mengikuti tren waktu; koreksi berisiko")
    print("     ikut membuang sinyal asli. Metode ini TIDAK DAPAT dipakai.")

r_lo, p_lo = stats.pearsonr(obs, y)
print(f"\nKorelasi luas vs observasi  : r = {r_lo:+.3f}, p = {p_lo:.4f}")
print(f"  -> {r_lo**2*100:.1f}% variasi luas dijelaskan jumlah observasi.")


UJI SYARAT KEABSAHAN KOREKSI
Korelasi observasi vs tahun : r = +0.174, p = 0.631
  -> tidak signifikan. Observasi tidak mengikuti tren waktu,
     sehingga koreksi tidak akan menghilangkan tren temporal asli.
     SYARAT TERPENUHI.

Korelasi luas vs observasi  : r = +0.788, p = 0.0067
  -> 62.2% variasi luas dijelaskan jumlah observasi.


3. MODEL REGRESI

In [14]:
print()
print("=" * 74)
print("MODEL REGRESI: Luas = a + b * Observasi")
print("=" * 74)

reg = stats.linregress(obs, y)
b, a = reg.slope, reg.intercept
obs_acuan = obs.mean()

# selang kepercayaan 95% untuk koefisien b
t_krit = stats.t.ppf(0.975, len(obs) - 2)
b_lo = b - t_krit * reg.stderr
b_hi = b + t_krit * reg.stderr

print(f"  a (intersep)   = {a:>12,.1f} ha")
print(f"  b (kemiringan) = {b:>12,.1f} ha per observasi")
print(f"  IK 95% untuk b = [{b_lo:,.1f} ; {b_hi:,.1f}]")
print(f"  R kuadrat      = {reg.rvalue**2:.3f}")
print(f"  nilai p        = {reg.pvalue:.4f}")
print(f"  observasi acuan= {obs_acuan:.1f}")


MODEL REGRESI: Luas = a + b * Observasi
  a (intersep)   =     55,303.3 ha
  b (kemiringan) =      1,059.8 ha per observasi
  IK 95% untuk b = [385.6 ; 1,733.9]
  R kuadrat      = 0.622
  nilai p        = 0.0067
  observasi acuan= 12.0


4. PENERAPAN KOREKSI

In [15]:
df["luas_terkoreksi"] = y - b * (obs - obs_acuan)
df["penyesuaian"] = df["luas_terkoreksi"] - y
df["delta_asli_pct"] = df[KOLOM_LUAS].pct_change() * 100
df["delta_koreksi_pct"] = df["luas_terkoreksi"].pct_change() * 100
df["persen_wilayah"] = df["luas_terkoreksi"] / LUAS_DIY * 100

print()
print("=" * 74)
print("HASIL KOREKSI")
print("=" * 74)
tampil = df[["tahun", KOLOM_OBS, KOLOM_LUAS, "delta_asli_pct",
             "luas_terkoreksi", "delta_koreksi_pct", "persen_wilayah"]]
tampil.columns = ["Tahun", "Obs", "Asli (ha)", "d% asli",
                  "Terkoreksi (ha)", "d% koreksi", "% DIY"]
print(tampil.round(1).to_string(index=False))


HASIL KOREKSI
 Tahun  Obs  Asli (ha)  d% asli  Terkoreksi (ha)  d% koreksi  % DIY
  2016  4.3    53500.4      NaN          61642.4         NaN   19.3
  2017  7.0    62526.7     16.9          67827.4        10.0   21.3
  2018 16.2    67507.8      8.0          63121.6        -6.9   19.8
  2019 22.4    73595.3      9.0          62572.4        -0.9   19.6
  2020 13.5    68308.9     -7.2          66750.9         6.7   21.0
  2021  8.5    67656.4     -1.0          71390.2         7.0   22.4
  2022  7.1    65023.8     -3.9          70269.0        -1.6   22.1
  2023 16.7    79881.1     22.8          74887.0         6.6   23.5
  2024 15.2    77437.3     -3.1          74115.3        -1.0   23.3
  2025  9.3    65018.5    -16.0          67880.0        -8.4   21.3


5. DIAGNOSTIK SETELAH KOREKSI

In [16]:
print()
print("=" * 74)
print("DIAGNOSTIK")
print("=" * 74)

r_sisa, _ = stats.pearsonr(obs, df["luas_terkoreksi"])
print(f"Korelasi luas terkoreksi vs observasi : r = {r_sisa:+.3f}")
print("  -> mendekati nol berarti artefak sudah hilang.")

sd_asli = df["delta_asli_pct"].std()
sd_kor  = df["delta_koreksi_pct"].std()
print(f"\nSimpangan baku perubahan tahunan:")
print(f"  sebelum koreksi : {sd_asli:5.1f}%")
print(f"  sesudah koreksi : {sd_kor:5.1f}%")
print(f"  penurunan ragam : {(1 - sd_kor/sd_asli)*100:5.1f}%")

tren = stats.linregress(th, df["luas_terkoreksi"])
print(f"\nTren linear setelah koreksi:")
print(f"  {tren.slope:+,.0f} ha per tahun "
      f"({tren.slope/df['luas_terkoreksi'].mean()*100:+.2f}% per tahun)")
print(f"  R kuadrat = {tren.rvalue**2:.3f}, p = {tren.pvalue:.4f}")

turun = df[df["delta_koreksi_pct"] < 0]
print(f"\nTahun yang masih menunjukkan penurunan: {len(turun)} dari {len(df)-1}")
if len(turun) > 0:
    print("  " + ", ".join(f"{int(r.tahun)} ({r.delta_koreksi_pct:+.1f}%)"
                           for r in turun.itertuples()))
    print("  Lahan terbangun secara fisik tidak berkurang; penurunan yang")
    print("  tersisa merupakan derau residual yang perlu dilaporkan.")


DIAGNOSTIK
Korelasi luas terkoreksi vs observasi : r = +0.000
  -> mendekati nol berarti artefak sudah hilang.

Simpangan baku perubahan tahunan:
  sebelum koreksi :  12.3%
  sesudah koreksi :   6.6%
  penurunan ragam :  46.6%

Tren linear setelah koreksi:
  +1,132 ha per tahun (+1.66% per tahun)
  R kuadrat = 0.537, p = 0.0159

Tahun yang masih menunjukkan penurunan: 5 dari 9
  2018 (-6.9%), 2019 (-0.9%), 2022 (-1.6%), 2024 (-1.0%), 2025 (-8.4%)
  Lahan terbangun secara fisik tidak berkurang; penurunan yang
  tersisa merupakan derau residual yang perlu dilaporkan.


6. NILAI UNTUK MODEL SISTEM DINAMIS

In [17]:
print()
print("=" * 74)
print("NILAI UNTUK MODEL SISTEM DINAMIS (tahun dasar 2025)")
print("=" * 74)
nilai_2025 = df.loc[df["tahun"] == 2025, "luas_terkoreksi"].iloc[0]
rddl = 1 - nilai_2025 / LUAS_DIY
print(f"  Lahan Terbangun (nilai awal)      = {nilai_2025:,.0f} ha")
print(f"  Luas Lahan Tersedia               = {LUAS_DIY:,} ha")
print(f"  Rasio Daya Dukung Lahan Referensi = {rddl:.4f}")


NILAI UNTUK MODEL SISTEM DINAMIS (tahun dasar 2025)
  Lahan Terbangun (nilai awal)      = 67,880 ha
  Luas Lahan Tersedia               = 318,580 ha
  Rasio Daya Dukung Lahan Referensi = 0.7869


7. EKSPOR

In [18]:
keluaran = df[["tahun", "n_citra", KOLOM_OBS, "cakupan_ha", KOLOM_LUAS,
               "luas_terkoreksi", "penyesuaian", "persen_wilayah"]]
output_csv_path = "/content/drive/MyDrive/Skripsi_Citra/Dynamic World - Built Up Area/Lahan_Terbangun_Terkoreksi_DIY_2016_2025.csv"
keluaran.to_csv(output_csv_path, index=False)
print()
print(f"Hasil disimpan: {output_csv_path}")


Hasil disimpan: /content/drive/MyDrive/Skripsi_Citra/Dynamic World - Built Up Area/Lahan_Terbangun_Terkoreksi_DIY_2016_2025.csv
